In [1]:
import os
import pandas as pd
import numpy as np
import cv2
import time
import uuid
from xml.etree.ElementTree import Element, SubElement, tostring
from xml.dom.minidom import parseString
from PIL import Image

In [2]:
IMAGES_PATH = os.path.join('TEAM MATES ASL')


In [3]:
if not os.path.exists(IMAGES_PATH):
    if os.name == 'posix':
        !mkdir -p {IMAGES_PATH}
    if os.name == 'nt':
        !mkdir {IMAGES_PATH}
for label in labels:
    path = os.path.join(IMAGES_PATH, label)
    if not os.path.exists(path):
        !mkdir {path}

NameError: name 'labels' is not defined

In [ ]:
for label in labels:
    cap = cv2.VideoCapture(0)
    print('collecting images {}'.format(label))
    time.sleep(5)
    for imgnum in range(number_imgs):
        print('collecting images {}'.format(imgnum))
        ret, frame = cap.read()
        imgname = os.path.join(IMAGES_PATH, label,label+'.'+'{}.jpg'.format(str(uuid.uuid1())))
        cv2.imwrite(imgname, frame)
        cv2.imshow('frame', frame)
        time.sleep(5)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
cap.release()
cv2.destroyAllWindows()

In [3]:
LABELING_PATH = os.path.join('labeling','labeling img')

In [4]:
if not os.path.exists(LABELING_PATH):
    !mkdir {LABELING_PATH}
    !git clone https://github.com/tzutalin/labelImg "{LABELING_PATH.replace(os.sep.'/')}"

In [7]:
if os.name == 'posix':
        !cd {LABELING_PATH} && make qt5py3
if os.name == 'nt':
        !cd {LABELING_PATH} && pyrcc5 -o libs/resources.py resources.qrc

In [7]:
!cd {LABELING_PATH} && python labelImg.py

Image:D:\sign-detection-small\project\images\ASL_TRAIN\A1.jpg -> Annotation:D:/sign-detection-small/project/images/ASL_TRAIN\A1.xml
Image:D:\sign-detection-small\project\images\ASL_TRAIN\A1.jpg -> Annotation:D:/sign-detection-small/project/images/ASL_TRAIN\A1.xml


In [1]:
import os
from PIL import Image
import xml.etree.ElementTree as ET
from xml.dom import minidom
import re

# ---------------- CONFIG ----------------
DATASET_DIR = r"D:\sign-detection\sign-detection\project\images\archive\asl_alphabet_test\asl_alphabet_test"#@train@#

# Path where a label_map will be created/updated (placed at dataset root)
LABEL_MAP_PATH = os.path.join(os.path.dirname(DATASET_DIR), "label_map.pbtxt")
# If you already have a label_map and want new classes appended, set APPEND_TO_EXISTING_LABELMAP = True
APPEND_TO_EXISTING_LABELMAP = True
# ----------------------------------------

def prettify_xml(elem):
    raw = ET.tostring(elem, 'utf-8')
    parsed = minidom.parseString(raw)
    return parsed.toprettyxml(indent="  ")

def create_pascal_voc_xml(image_path, class_name):
    try:
        img = Image.open(image_path)
    except Exception as e:
        print(f"⚠️ Cannot open {image_path}: {e}")
        return False

    width, height = img.size
    depth = 3  # assuming RGB images

    # Build XML
    annotation = ET.Element('annotation')

    folder = ET.SubElement(annotation, 'folder')
    folder.text = class_name  # class folder name

    filename = ET.SubElement(annotation, 'filename')
    filename.text = os.path.basename(image_path)

    size = ET.SubElement(annotation, 'size')
    ET.SubElement(size, 'width').text = str(width)
    ET.SubElement(size, 'height').text = str(height)
    ET.SubElement(size, 'depth').text = str(depth)

    # Object (whole-image bbox)
    obj = ET.SubElement(annotation, 'object')
    ET.SubElement(obj, 'name').text = class_name
    ET.SubElement(obj, 'pose').text = 'Unspecified'
    ET.SubElement(obj, 'truncated').text = '0'
    ET.SubElement(obj, 'difficult').text = '0'

    bndbox = ET.SubElement(obj, 'bndbox')
    ET.SubElement(bndbox, 'xmin').text = '0'
    ET.SubElement(bndbox, 'ymin').text = '0'
    ET.SubElement(bndbox, 'xmax').text = str(width)
    ET.SubElement(bndbox, 'ymax').text = str(height)

    xml_str = prettify_xml(annotation)
    xml_path = os.path.splitext(image_path)[0] + ".xml"
    with open(xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_str)
    return True

def find_existing_labelmap_info(path):
    """Return (existing_names_set, max_id)"""
    if not os.path.exists(path):
        return set(), 0
    text = open(path, 'r', encoding='utf-8').read()
    # find all name: 'X' entries
    names = set(re.findall(r"name:\s*['\"]([^'\"]+)['\"]", text))
    ids = [int(x) for x in re.findall(r"id:\s*([0-9]+)", text)]
    max_id = max(ids) if ids else 0
    return names, max_id

def append_labelmap(path, new_classes, start_id):
    with open(path, 'a', encoding='utf-8') as f:
        for i, cls in enumerate(new_classes, start=start_id):
            f.write("item {\n")
            f.write(f"  id: {i}\n")
            f.write(f"  name: '{cls}'\n")
            f.write("}\n\n")

def write_labelmap(path, classes, start_id=1):
    with open(path, 'w', encoding='utf-8') as f:
        for i, cls in enumerate(classes, start=start_id):
            f.write("item {\n")
            f.write(f"  id: {i}\n")
            f.write(f"  name: '{cls}'\n")
            f.write("}\n\n")

# ---------------- MAIN ----------------
# 1) discover class folders (sorted)
classes = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]
classes.sort()
print(f"Found {len(classes)} class folders (examples): {classes[:6]} ...")

# 2) Create XML files next to each image
total = 0
for class_name in classes:
    class_path = os.path.join(DATASET_DIR, class_name)
    print(f"Processing class folder: {class_path}")
    for fname in os.listdir(class_path):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        image_path = os.path.join(class_path, fname)
        ok = create_pascal_voc_xml(image_path, class_name)
        if ok:
            total += 1
print(f"✅ Created/updated XMLs for {total} images.")

# 3) Generate or append label_map.pbtxt
existing_names, max_id = find_existing_labelmap_info(LABEL_MAP_PATH)
to_add = [c for c in classes if c not in existing_names]
if os.path.exists(LABEL_MAP_PATH) and APPEND_TO_EXISTING_LABELMAP:
    start_id = max_id + 1
    if to_add:
        append_labelmap(LABEL_MAP_PATH, to_add, start_id=start_id)
        print(f"Appended {len(to_add)} classes to existing label_map at {LABEL_MAP_PATH}.")
    else:
        print("No new classes to append to existing label_map.")
else:
    # create a fresh label map starting id=1 (or keep APPEND behavior off)
    write_labelmap(LABEL_MAP_PATH, classes, start_id=1)
    print(f"Wrote new label_map with {len(classes)} classes to {LABEL_MAP_PATH}.")

print("All done.")


Found 0 class folders (examples): [] ...
✅ Created/updated XMLs for 0 images.


NameError: name 'LABEL_MAP_PATH' is not defined